# NB54 — 63k Genis Veri: Final Degerlendirme (Test Seti Ilk ve Son Kez Aciliyor)

Plan: `docs/PLAN_63K_ENTEGRASYON.md` ADIM 5. NB53'un champion recete v2'sini
(Optuna-LGBM + native_nan + scale_pos_weight + top200 feature + meta-predictor
dahil + no_fe + isotonic kalibrasyon + threshold=0.46, CV F1=0.9907) sabit
recete olarak alip:

1. **Stratified train/test split** (`TEST_SIZE=0.2`, config.py) -- bu calisma boyunca
   ilk kez gercek hold-out ayriliyor. Simdiye kadar NB50-53 sadece CV kullandi.
2. **Final fit** (train'de, recete sabit) + **test'te tek seferlik degerlendirme**.
3. **CV-test farki** (protokol durustlugu kaniti) -- NB53'un CV F1=0.9907'i ile
   test F1'i arasindaki fark raporlanir.
4. **Gen-holdout final skoru** (GroupKFold, `base__hugo`) -- rastgele split ile
   farki, gercek genelleme gucu.
5. **SHAP analizi** -- 63k'nin avantaji: sutunlar acik isimli, yorumlanabilir.
   Top-30 feature + biyolojik yorum.
6. **Hata analizi** -- FP/FN'ler gen bazinda gruplanir.
7. **Model kaydi** -- `models/v31_63k/` (model + threshold + feature listesi + imputer).

**Test seti bu notebook disinda ASLA kullanilmayacak** -- tek seferlik degerlendirme.

In [1]:
# Cell 1: Imports & Config
import os, sys, json, time, warnings
import torch  # ONCE torch import et -- macOS SIGSEGV onlemi (NB52/NB53'te dogrulandi)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, TEST_SIZE
from src import columns_63k as C63
from src.metrics import optimize_threshold, compute_all_metrics

np.random.seed(SEED)

from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef, average_precision_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
import shap
import joblib

PARQUET_DIR = os.path.join(PROJECT_ROOT, 'data', '63k_genis')
RESULTS_BASELINE_DIR = os.path.join(PROJECT_ROOT, 'results', 'v33_63k_baseline')
RESULTS_OPT_DIR = os.path.join(PROJECT_ROOT, 'results', 'v34_63k_optimization')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v35_63k_final')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models', 'v31_63k')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MISSENSE_PARQUET = os.path.join(PARQUET_DIR, 'missense_63k.parquet')
df = pd.read_parquet(MISSENSE_PARQUET)
print('missense_63k.parquet yuklendi:', df.shape)

with open(os.path.join(RESULTS_BASELINE_DIR, 'nb52_champion_feature_list.json')) as f:
    champion_features = json.load(f)
with open(os.path.join(RESULTS_OPT_DIR, 'nb53_champion_recipe_v2.json')) as f:
    recipe_v2 = json.load(f)

FEATURE_COLS = champion_features['feature_cols']
CAT_COLS = champion_features['cat_cols']
BEST_LGBM_PARAMS = recipe_v2['optuna_params']
FINAL_THRESHOLD = recipe_v2['final_threshold']
CALIB_METHOD = recipe_v2['calibration_method']
NB53_CV_F1 = recipe_v2['final_cv_f1']

print(f"Sabit recete: lgbm + Optuna params + native_nan + scale_pos_weight + top200 ({len(FEATURE_COLS)} feature)")
print(f"Kalibrasyon: {CALIB_METHOD}, threshold: {FINAL_THRESHOLD:.2f}")
print(f"NB53 referans CV F1: {NB53_CV_F1:.4f}")

y_full = df['Label'].astype(int)
groups_full = df[C63.GENE_GROUP_COL]
print(f'n={len(df)}, prevalans={y_full.mean():.4f}, TEST_SIZE={TEST_SIZE}, SEED={SEED}')

missense_63k.parquet yuklendi: (60970, 533)
Sabit recete: lgbm + Optuna params + native_nan + scale_pos_weight + top200 (200 feature)
Kalibrasyon: isotonic, threshold: 0.46
NB53 referans CV F1: 0.9907
n=60970, prevalans=0.3727, TEST_SIZE=0.2, SEED=42


## 1. Stratified Train/Test Split — Test Seti Ilk Kez Aciliyor

`TEST_SIZE` (`config.py`'dan), stratified. Bu split bu notebook'un disinda
hicbir yerde kullanilmadi -- NB50-53 sadece CV yapti. Split sonrasi imputer/encoder
**yalnizca train'de fit edilir** (CLAUDE.md sozlesme #1).

In [2]:
# Cell 2: Train/test split (ilk ve tek kez)
train_idx, test_idx = train_test_split(
    df.index, test_size=TEST_SIZE, stratify=y_full, random_state=SEED)

df_train = df.loc[train_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)
y_train = df_train['Label'].astype(int)
y_test = df_test['Label'].astype(int)
groups_train = df_train[C63.GENE_GROUP_COL]

floor_train = C63.floor_f1(y_train.mean())
floor_test = C63.floor_f1(y_test.mean())
spw_train = (y_train == 0).sum() / (y_train == 1).sum()

print(f'Train: n={len(df_train)}, prevalans={y_train.mean():.4f}, floor-F1={floor_train:.4f}')
print(f'Test:  n={len(df_test)}, prevalans={y_test.mean():.4f}, floor-F1={floor_test:.4f}')
print(f'scale_pos_weight (train)={spw_train:.4f}')

with open(os.path.join(RESULTS_DIR, 'nb54_split_indices.json'), 'w') as f:
    json.dump({'train_idx': [int(i) for i in train_idx], 'test_idx': [int(i) for i in test_idx],
               'test_size': TEST_SIZE, 'seed': SEED}, f)

Train: n=48776, prevalans=0.3727, floor-F1=0.5430
Test:  n=12194, prevalans=0.3726, floor-F1=0.5430
scale_pos_weight (train)=1.6832


## 2. Final Fit (Train) + Tek Seferlik Test Degerlendirmesi

Recete sabit (NB53 champion v2): Optuna-LGBM + native_nan (impute yok) +
scale_pos_weight + top200 feature + isotonic kalibrasyon + threshold=0.46.
Imputer/encoder yok cunku native_nan stratejisi kullaniliyor (LGBM NaN'i
dogrudan isliyor); kategorik sutunlar train-only OrdinalEncoder ile encode edilir.

In [3]:
# Cell 3: Train-only encode + final model fit
def encode_categoricals_train_only(X_tr, X_te, cat_cols):
    if not cat_cols:
        return X_tr, X_te
    X_tr = X_tr.copy(); X_te = X_te.copy()
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_tr[cat_cols] = X_tr[cat_cols].astype(str).fillna('__NA__')
    X_te[cat_cols] = X_te[cat_cols].astype(str).fillna('__NA__')
    X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols])
    X_te[cat_cols] = enc.transform(X_te[cat_cols])
    return X_tr, X_te, enc

X_train_raw = df_train[FEATURE_COLS].copy()
X_test_raw = df_test[FEATURE_COLS].copy()
X_train, X_test, cat_encoder = encode_categoricals_train_only(X_train_raw, X_test_raw, CAT_COLS)

final_model = lgb.LGBMClassifier(
    n_estimators=BEST_LGBM_PARAMS.get('n_estimators', 300),
    learning_rate=BEST_LGBM_PARAMS.get('learning_rate', 0.05),
    num_leaves=BEST_LGBM_PARAMS.get('num_leaves', 31),
    max_depth=BEST_LGBM_PARAMS.get('max_depth', -1),
    min_child_samples=BEST_LGBM_PARAMS.get('min_child_samples', 20),
    subsample=BEST_LGBM_PARAMS.get('subsample', 1.0),
    colsample_bytree=BEST_LGBM_PARAMS.get('colsample_bytree', 1.0),
    reg_alpha=BEST_LGBM_PARAMS.get('reg_alpha', 0.0),
    reg_lambda=BEST_LGBM_PARAMS.get('reg_lambda', 0.0),
    random_state=SEED, verbosity=-1, scale_pos_weight=spw_train)

t0 = time.time()
final_model.fit(X_train, y_train)
fit_elapsed = time.time() - t0
print(f'Final model egitildi ({fit_elapsed:.1f}s)')

train_proba_raw = final_model.predict_proba(X_train)[:, 1]
test_proba_raw = final_model.predict_proba(X_test)[:, 1]

Final model egitildi (3.7s)


In [4]:
# Cell 4: Isotonic kalibrasyon (train'de fit, test'e transform -- NB53 sozlesmesiyle ayni)
if CALIB_METHOD == 'isotonic':
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(train_proba_raw, y_train.values)
    train_proba = calibrator.predict(train_proba_raw)
    test_proba = calibrator.predict(test_proba_raw)
else:
    calibrator = None
    train_proba = train_proba_raw
    test_proba = test_proba_raw

print(f'Kalibrasyon uygulandi: {CALIB_METHOD}')

train_pred = (train_proba >= FINAL_THRESHOLD).astype(int)
test_pred = (test_proba >= FINAL_THRESHOLD).astype(int)
train_metrics = compute_all_metrics(y_train.values, train_pred, train_proba)
test_metrics = compute_all_metrics(y_test.values, test_pred, test_proba)

print()
print('=== TRAIN METRIKLERI (threshold=%.2f) ===' % FINAL_THRESHOLD)
for k, v in train_metrics.items():
    print(f'{k}: {v}')
print()
print('=== TEST METRIKLERI (threshold=%.2f, ILK VE TEK KEZ) ===' % FINAL_THRESHOLD)
for k, v in test_metrics.items():
    print(f'{k}: {v}')

overfit_gap = train_metrics['f1'] - test_metrics['f1']
print()
print(f'Train-test F1 farki (overfit gap) = {overfit_gap:+.4f}')

cm = confusion_matrix(y_test, (test_proba >= FINAL_THRESHOLD).astype(int))
print()
print('=== TEST CONFUSION MATRIX ===')
print(f'TN={cm[0,0]}  FP={cm[0,1]}')
print(f'FN={cm[1,0]}  TP={cm[1,1]}')

print()
print(f'Test floor-F1 (hep pathogenic de) = {floor_test:.4f}')
print(f'Test F1 - floor farki = {test_metrics["f1"] - floor_test:+.4f}')

Kalibrasyon uygulandi: isotonic

=== TRAIN METRIKLERI (threshold=0.46) ===
f1: 1.0
auc_roc: 1.0
auc_pr: 1.0
mcc: 1.0
precision: 1.0
recall: 1.0
specificity: 1.0
balanced_accuracy: 1.0
cohens_kappa: 1.0

=== TEST METRIKLERI (threshold=0.46, ILK VE TEK KEZ) ===
f1: 0.990210097899021
auc_roc: 0.9944688104114886
auc_pr: 0.988940559317849
mcc: 0.9843921008720925
precision: 0.9898834396305256
recall: 0.9905369718309859
specificity: 0.9939869281045751
balanced_accuracy: 0.9922619499677805
cohens_kappa: 0.9843919646344463

Train-test F1 farki (overfit gap) = +0.0098

=== TEST CONFUSION MATRIX ===
TN=7604  FP=46
FN=43  TP=4501

Test floor-F1 (hep pathogenic de) = 0.5430
Test F1 - floor farki = +0.4473


## 3. CV-Test Farki — Protokol Durustlugu Kaniti

NB53'un 5-fold CV F1'i (0.9907, tum veri uzerinde OOF) ile bu notebook'ta
ayrilan gercek hold-out test F1'i karsilastirilir. Buyuk fark (>0.01-0.02)
CV'nin iyimser oldugunu, kucuk fark protokolun guvenilir oldugunu gosterir.

In [5]:
# Cell 5: CV-test farki
cv_test_gap = NB53_CV_F1 - test_metrics['f1']
print(f'NB53 CV F1 (OOF, tum veri)     = {NB53_CV_F1:.4f}')
print(f'NB54 Test F1 (gercek hold-out) = {test_metrics["f1"]:.4f}')
print(f'Fark (CV - test)               = {cv_test_gap:+.4f}')

if abs(cv_test_gap) <= 0.01:
    protocol_verdict = 'SAGLAM -- CV tahmini test performansini guvenilir yansitiyor'
elif abs(cv_test_gap) <= 0.03:
    protocol_verdict = 'KABUL EDILEBILIR -- kucuk sapma, izlenmeli'
else:
    protocol_verdict = 'SUPHELI -- CV iyimser, olasi sizinti/overfit arastirilmali'
print(f'Protokol durustlugu karari: {protocol_verdict}')

NB53 CV F1 (OOF, tum veri)     = 0.9907
NB54 Test F1 (gercek hold-out) = 0.9902
Fark (CV - test)               = +0.0005
Protokol durustlugu karari: SAGLAM -- CV tahmini test performansini guvenilir yansitiyor


## 4. Gen-Holdout Final Skoru — Gercek Genelleme Gucu

`base__hugo` (gen adi) grubuna gore GroupKFold ile ayni train uzerinde egitilen
modelin gen-bazli tahmin performansi. Rastgele split (yukaridaki test) ile
gen-holdout arasindaki fark, modelin gen ezberi yapip yapmadigini gosterir
(NB51'de bu fark 0.0056 ile onemsiz bulunmustu -- burada teyit edilir).

In [6]:
# Cell 6: Gen-holdout (GroupKFold, train uzerinde) vs rastgele test farki
N_GENE_SPLITS = 5
gkf = GroupKFold(n_splits=N_GENE_SPLITS)
gene_oof = np.zeros(len(df_train))

for tr_idx, val_idx in gkf.split(X_train, y_train, groups=groups_train):
    X_tr_g, X_val_g = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr_g = y_train.iloc[tr_idx]
    m = lgb.LGBMClassifier(
        n_estimators=BEST_LGBM_PARAMS.get('n_estimators', 300),
        learning_rate=BEST_LGBM_PARAMS.get('learning_rate', 0.05),
        num_leaves=BEST_LGBM_PARAMS.get('num_leaves', 31),
        max_depth=BEST_LGBM_PARAMS.get('max_depth', -1),
        min_child_samples=BEST_LGBM_PARAMS.get('min_child_samples', 20),
        subsample=BEST_LGBM_PARAMS.get('subsample', 1.0),
        colsample_bytree=BEST_LGBM_PARAMS.get('colsample_bytree', 1.0),
        reg_alpha=BEST_LGBM_PARAMS.get('reg_alpha', 0.0),
        reg_lambda=BEST_LGBM_PARAMS.get('reg_lambda', 0.0),
        random_state=SEED, verbosity=-1, scale_pos_weight=spw_train)
    m.fit(X_tr_g, y_tr_g)
    gene_oof[val_idx] = m.predict_proba(X_val_g)[:, 1]

gene_f1 = f1_score(y_train, (gene_oof >= FINAL_THRESHOLD).astype(int))
random_test_f1 = test_metrics['f1']
gene_gap = random_test_f1 - gene_f1

print(f'Gen-holdout F1 (GroupKFold, train)     = {gene_f1:.4f}')
print(f'Rastgele hold-out test F1              = {random_test_f1:.4f}')
print(f'Fark (rastgele - gen-holdout)          = {gene_gap:+.4f}')

if gene_gap > 0.10:
    gene_verdict = 'CIDDI GEN EZBERI -- gen-holdout birincil metrik olmali'
elif gene_gap > 0.03:
    gene_verdict = 'ORTA DUZEY GEN EZBERI -- izlenmeli'
else:
    gene_verdict = 'BELIRGIN DEGIL -- NB51 bulgusuyla tutarli'
print(f'Karar: {gene_verdict}')

Gen-holdout F1 (GroupKFold, train)     = 0.9881
Rastgele hold-out test F1              = 0.9902
Fark (rastgele - gen-holdout)          = +0.0021
Karar: BELIRGIN DEGIL -- NB51 bulgusuyla tutarli


## 5. SHAP Analizi — Top-30 Feature + Biyolojik Yorum

63k verisinin avantaji: sutun isimleri acik (`cadd__score`, `revel__score`,
`gerp__gerp_rs` vb.), SHAP degerleri dogrudan yorumlanabilir. Beklenti:
REVEL/AlphaMissense/CADD/konservasyon (phyloP, GERP) skorlari tepede olmali --
degilse siniflandirma mantigi supheli.

In [7]:
# Cell 7: SHAP top-30 (TreeExplainer, test seti uzerinde)
explainer = shap.TreeExplainer(final_model)
shap_sample = X_test if len(X_test) <= 5000 else X_test.sample(5000, random_state=SEED)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # positive class

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=FEATURE_COLS).sort_values(ascending=False)
top30 = shap_importance.head(30)

print('=== TOP-30 FEATURE (mean |SHAP|) ===')
print(top30.to_string())

fig, ax = plt.subplots(figsize=(9, 10))
top30.iloc[::-1].plot.barh(ax=ax)
ax.set_xlabel('mean |SHAP value|')
ax.set_title('NB54 - Top-30 Feature Importance (SHAP, test seti)')
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'nb54_shap_top30.png'), dpi=120)
plt.close(fig)

top30.to_csv(os.path.join(RESULTS_DIR, 'nb54_shap_top30.csv'))
print('SHAP top-30 grafigi ve CSV kaydedildi.')

=== TOP-30 FEATURE (mean |SHAP|) ===
ditto__score                          4.414764
metarnn__score                        1.246201
allofus250k__gvs_max_af               0.771111
gnomad__af                            0.760769
metarnn__rank_score                   0.677944
mistic__score                         0.618237
alphamissense__am_pathogenicity       0.551166
vest__pval                            0.422587
flank_seq__ref_seq                    0.420289
bayesdel__bayesdel_addAF_score        0.398701
bayesdel__bayesdel_addAF_rankscore    0.354236
gmvp__score                           0.346699
gnomad__af_sas                        0.342951
vest__score                           0.341942
dbsnp__rsid                           0.299372
clinpred__score                       0.295495
gmvp__rank_score                      0.285532
mutpred1__mutpred_general_score       0.277370
allofus250k__gvs_oth_af               0.205801
primateai__primateai_rankscore        0.192199
regeneron__ALL_AF      

## 6. Hata Analizi — FP/FN Gen Bazinda Gruplama

Test setindeki yanlis siniflandirmalar (FP: benign iken pathogenic tahmin,
FN: pathogenic iken benign tahmin) `base__hugo` gen sutununa gore gruplanir.
Hangi genlerde model sistematik olarak zorlaniyor?

In [8]:
# Cell 8: FP/FN gen bazinda hata analizi
test_pred = (test_proba >= FINAL_THRESHOLD).astype(int)
error_df = pd.DataFrame({
    'gene': df_test[C63.GENE_GROUP_COL].values,
    'y_true': y_test.values,
    'y_pred': test_pred,
    'proba': test_proba,
})
error_df['error_type'] = np.select(
    [ (error_df.y_true == 0) & (error_df.y_pred == 1),
      (error_df.y_true == 1) & (error_df.y_pred == 0) ],
    ['FP', 'FN'], default='correct')

fp_by_gene = error_df[error_df.error_type == 'FP'].groupby('gene').size().sort_values(ascending=False)
fn_by_gene = error_df[error_df.error_type == 'FN'].groupby('gene').size().sort_values(ascending=False)

print('=== En cok FP ureten 15 gen ===')
print(fp_by_gene.head(15).to_string())
print()
print('=== En cok FN ureten 15 gen ===')
print(fn_by_gene.head(15).to_string())

n_fp, n_fn = (error_df.error_type == 'FP').sum(), (error_df.error_type == 'FN').sum()
print()
print(f'Toplam FP={n_fp}, FN={n_fn} (test n={len(error_df)})')

error_df.to_csv(os.path.join(RESULTS_DIR, 'nb54_test_errors.csv'), index=False)
fp_by_gene.to_csv(os.path.join(RESULTS_DIR, 'nb54_fp_by_gene.csv'))
fn_by_gene.to_csv(os.path.join(RESULTS_DIR, 'nb54_fn_by_gene.csv'))

=== En cok FP ureten 15 gen ===
gene
RYR1       2
KAT6B      2
MSH2       2
BRCA1      2
GAA        1
ADA        1
HNF1A      1
PSAP       1
C9         1
GABRA1     1
CTLA4      1
NEDD4L     1
HNRNPU     1
SCN11A     1
SYNGAP1    1

=== En cok FN ureten 15 gen ===
gene
TP53        2
DYNC2H1     2
ACADM       2
F8          1
MYO15A      1
TTLL5       1
GJB2        1
CYP24A1     1
ASPA        1
MYH7        1
SERPINC1    1
UGP2        1
HBA2        1
HBB         1
RARS1       1

Toplam FP=46, FN=43 (test n=12194)


## 7. Model Kaydi — `models/v31_63k/`

Final model, kalibratör, feature listesi, kategorik encoder ve threshold birlikte
serialize edilir (deploy/inference icin tek paket).

In [9]:
# Cell 9: Model + artifact kaydi
joblib.dump(final_model, os.path.join(MODELS_DIR, 'nb54_final_lgbm_model.joblib'))
if calibrator is not None:
    joblib.dump(calibrator, os.path.join(MODELS_DIR, 'nb54_isotonic_calibrator.joblib'))
joblib.dump(cat_encoder, os.path.join(MODELS_DIR, 'nb54_cat_encoder.joblib'))

model_manifest = {
    'model_family': 'lgbm',
    'optuna_params': BEST_LGBM_PARAMS,
    'feature_cols': FEATURE_COLS,
    'cat_cols': CAT_COLS,
    'calibration_method': CALIB_METHOD,
    'final_threshold': float(FINAL_THRESHOLD),
    'scale_pos_weight_train': float(spw_train),
    'seed': SEED,
    'test_size': TEST_SIZE,
    'test_metrics': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in test_metrics.items()},
    'train_metrics': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in train_metrics.items()},
    'cv_test_gap': float(cv_test_gap),
    'gene_holdout_f1': float(gene_f1),
    'gene_gap': float(gene_gap),
    'floor_f1_test': float(floor_test),
}
with open(os.path.join(MODELS_DIR, 'nb54_model_manifest.json'), 'w') as f:
    json.dump(model_manifest, f, indent=2, ensure_ascii=False)

with open(os.path.join(RESULTS_DIR, 'nb54_final_results.json'), 'w') as f:
    json.dump(model_manifest, f, indent=2, ensure_ascii=False)

print('Model artifact seti kaydedildi:')
for fn in ['nb54_final_lgbm_model.joblib', 'nb54_isotonic_calibrator.joblib',
           'nb54_cat_encoder.joblib', 'nb54_model_manifest.json']:
    p = os.path.join(MODELS_DIR, fn)
    print(f'  {fn}: {"OK" if os.path.exists(p) else "EKSIK"}')

Model artifact seti kaydedildi:
  nb54_final_lgbm_model.joblib: OK
  nb54_isotonic_calibrator.joblib: OK
  nb54_cat_encoder.joblib: OK
  nb54_model_manifest.json: OK


## 8. Ozet + PDF Rapor

In [10]:
# Cell 10: Ozet grafik + PDF rapor
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
metrics_to_plot = ['f1', 'mcc', 'auc_pr', 'auc_roc'] if 'auc_roc' in test_metrics else ['f1', 'mcc', 'auc_pr']
metrics_to_plot = [m for m in metrics_to_plot if m in train_metrics and m in test_metrics]
x = np.arange(len(metrics_to_plot))
w = 0.35
axes[0].bar(x - w/2, [train_metrics[m] for m in metrics_to_plot], w, label='Train')
axes[0].bar(x + w/2, [test_metrics[m] for m in metrics_to_plot], w, label='Test (hold-out)')
axes[0].axhline(floor_test, color='red', linestyle='--', label=f'Floor-F1(test)={floor_test:.3f}')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_to_plot)
axes[0].set_title('Train vs Test Metrikleri')
axes[0].legend()

axes[1].bar(['Rastgele test', 'Gen-holdout'], [random_test_f1, gene_f1], color=['C0', 'C1'])
axes[1].axhline(floor_test, color='red', linestyle='--')
axes[1].set_title('Rastgele vs Gen-Holdout F1')
axes[1].set_ylim(0, 1.05)

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'nb54_final_summary.png'), dpi=120)
plt.close(fig)

from fpdf import FPDF

class NB54Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB54 - 63k Final Degerlendirme Raporu', ln=True, align='C')
        self.ln(2)

    def section(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 8, title, ln=True)
        self.set_font('Helvetica', '', 9)

    def kv_table(self, d):
        for k, v in d.items():
            self.cell(0, 6, f'{k}: {v}', ln=True)
        self.ln(2)

report = NB54Report()
report.add_page()

report.section('1. Split Bilgisi')
report.kv_table({'test_size': TEST_SIZE, 'seed': SEED, 'train_n': len(df_train), 'test_n': len(df_test),
                  'train_prevalence': f'{y_train.mean():.4f}', 'test_prevalence': f'{y_test.mean():.4f}'})

report.section('2. Final Test Metrikleri')
report.kv_table({k: v for k, v in test_metrics.items()})

report.section('3. Train Metrikleri (Overfit Kontrolu)')
report.kv_table({k: v for k, v in train_metrics.items()})
report.cell(0, 6, f'Overfit gap (train F1 - test F1) = {overfit_gap:+.4f}', ln=True)

report.section('4. CV-Test Farki (Protokol Durustlugu)')
report.kv_table({'nb53_cv_f1': f'{NB53_CV_F1:.4f}', 'nb54_test_f1': f'{test_metrics["f1"]:.4f}',
                  'fark': f'{cv_test_gap:+.4f}', 'karar': protocol_verdict})

report.section('5. Gen-Holdout Genelleme')
report.kv_table({'gene_holdout_f1': f'{gene_f1:.4f}', 'random_test_f1': f'{random_test_f1:.4f}',
                  'fark': f'{gene_gap:+.4f}', 'karar': gene_verdict})

report.section('6. Confusion Matrix (Test)')
report.kv_table({'TN': int(cm[0,0]), 'FP': int(cm[0,1]), 'FN': int(cm[1,0]), 'TP': int(cm[1,1])})

report.section('7. Top-10 SHAP Feature')
for feat, val in top30.head(10).items():
    report.cell(0, 5, f'{feat}: {val:.4f}', ln=True)

REPORT_PATH = os.path.join(REPORTS_DIR, 'nb54_final_report.pdf')
report.output(REPORT_PATH)
print(f'PDF rapor yazildi: {REPORT_PATH}')

print()
print('=== NB54 TAMAMLANDI ===')
print(f'Final test F1={test_metrics["f1"]:.4f} (floor={floor_test:.4f}, +{test_metrics["f1"]-floor_test:.4f})')
print(f'CV-test farki={cv_test_gap:+.4f} ({protocol_verdict})')
print(f'Gen-ezberi farki={gene_gap:+.4f} ({gene_verdict})')

PDF rapor yazildi: /Users/tefe/teknofest_model/teknofest_model/reports/nb54_final_report.pdf

=== NB54 TAMAMLANDI ===
Final test F1=0.9902 (floor=0.5430, +0.4473)
CV-test farki=+0.0005 (SAGLAM -- CV tahmini test performansini guvenilir yansitiyor)
Gen-ezberi farki=+0.0021 (BELIRGIN DEGIL -- NB51 bulgusuyla tutarli)


## Sonraki Adim

NB54 ile 63k genis veri hattinin optimizasyon+final degerlendirme dongusu
tamamlanir. `progress.md`'ye NB54 bolumu eklenmeli; `to-do.md` guncellenmeli.
Yarisma verisi (asil teslim hatti, NB12-NB48) ile 63k calismasi (bu hat) ayri
rejimler olarak kalmaya devam ediyor -- recete transferi yok (bkz. NB52/NB53
bulgulari).